In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "workspace"
SCHEMA = "default"

bronze_txns = spark.table(f"{CATALOG}.{SCHEMA}.bronze_transactions")
bronze_accts = spark.table(f"{CATALOG}.{SCHEMA}.bronze_accounts")

# 1. Deduplicate by txn_id (keep latest ingestion)
dedup_window = Window.partitionBy("txn_id").orderBy(F.col("_ingested_at").desc())
deduped = (
    bronze_txns
    .withColumn("_rn", F.row_number().over(dedup_window))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

# 2. Standardize dates, amounts, and FX rates to USD (ANSI-safe expressions)
fx_expr = (
    F.when(F.col("currency") == "USD", 1.00)
    .when(F.col("currency") == "EUR", 1.08)
    .when(F.col("currency") == "GBP", 1.27)
    .when(F.col("currency") == "CAD", 0.73)
    .otherwise(None)
)

parsed = (
    deduped
    .withColumn(
        "clean_date",
        F.expr("coalesce(try_to_date(txn_date, 'yyyy-MM-dd'), try_to_date(txn_date, 'MM/dd/yyyy'))")
    )
    .withColumn("clean_amount", F.expr("try_cast(amount as double)"))
    .withColumn("fx_rate", fx_expr)
    .withColumn("amount_usd", F.round(F.col("clean_amount") * F.col("fx_rate"), 2))
    .join(bronze_accts.select("account_id", "account_name", "account_type", "department"), on="account_id", how="left")
)

# 3. Classify data quality defects for Quarantine vs. Clean Silver
evaluated = parsed.withColumn(
    "quarantine_reason",
    F.when(F.col("status") != "POSTED", "NON_POSTED_STATUS")
    .when(F.col("clean_date").isNull(), "INVALID_DATE")
    .when(F.col("clean_amount").isNull() | (F.col("clean_amount") <= 0), "INVALID_AMOUNT")
    .when(F.col("account_name").isNull(), "UNMAPPED_ACCOUNT")
    .when(F.col("fx_rate").isNull(), "UNSUPPORTED_CURRENCY")
    .otherwise(F.lit(None))
)

quarantine_df = evaluated.filter(F.col("quarantine_reason").isNotNull())
valid_df = evaluated.filter(F.col("quarantine_reason").isNull()).drop("quarantine_reason")

# 4. Statistical Anomaly Detection (Z-Score per Account)
acct_window = Window.partitionBy("account_id")
silver_df = (
    valid_df
    .withColumn("acct_mean_usd", F.avg("amount_usd").over(acct_window))
    .withColumn("acct_std_usd", F.stddev("amount_usd").over(acct_window))
    .withColumn(
        "z_score",
        F.when(
            F.col("acct_std_usd") > 0,
            F.round((F.col("amount_usd") - F.col("acct_mean_usd")) / F.col("acct_std_usd"), 2)
        ).otherwise(F.lit(0.0))
    )
    .withColumn("is_anomaly", F.abs(F.col("z_score")) >= 3.0)
    .withColumn("txn_month", F.date_trunc("month", F.col("clean_date")).cast("date"))
    .select(
        "txn_id", "clean_date", "txn_month", "account_id", "account_name",
        "account_type", "department", "vendor_name", "currency",
        "clean_amount", "amount_usd", "z_score", "is_anomaly", "_ingested_at"
    )
)

# 5. Persist Silver & Quarantine Delta tables
quarantine_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.silver_quarantine_transactions")

silver_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.silver_transactions")

print(f"Silver Clean Rows : {silver_df.count():,}")
print(f"Quarantined Rows  : {quarantine_df.count():,}")
print(f"Flagged Anomalies : {silver_df.filter(F.col('is_anomaly')).count():,}")